In [ ]:
import pandas 
import pathlib
import ipywidgets
from ipyfilechooser import FileChooser

import widget_code_cell


In [ ]:
# Global
DF_FILES = pandas.DataFrame(columns=["filename", "size_gb", "modified", "full_path"])

In [ ]:
# Functions
def file_metadata(path: pathlib.Path):
    return {
        "filename": path.name,
        "size_gb": round(path.stat().st_size / 1e9, 2),
        "modified": pandas.to_datetime(path.stat().st_mtime, unit="s"),
        "full_path": path.resolve(),
    }

In [ ]:
# Widgets

fc_path_input = FileChooser(
    path = pathlib.Path('.').resolve(),
    show_only_dirs=True,
    title='Select Directory',
    layout=ipywidgets.Layout(width="80%"),
    )

clear_button = ipywidgets.Button(
    description="Clear",
    button_style="danger",
    layout=ipywidgets.Layout(width="120px")
)

add_output = ipywidgets.Output()

file_selector = ipywidgets.SelectMultiple(
    options=DF_FILES["filename"].tolist(),
    description="Files:",
    layout=ipywidgets.Layout(width="80%", height="200px")
)

load_button = ipywidgets.Button(
    description="Load",
    button_style="info",
    layout=ipywidgets.Layout(width="120px")
)

load_output = ipywidgets.Output()

In [ ]:
# Callback
def refresh_selector():
    file_selector.options = DF_FILES["filename"].tolist()


def on_clear_clicked(b):
    global DF_FILES
    DF_FILES = pandas.DataFrame(columns=["filename", "size_gb", "modified", "full_path"])
    refresh_selector()  


def on_load_clicked(b):
    load_output.clear_output()
    with load_output:
        selected = list(file_selector.value)
        if not selected:
            print("No files selected.")
            return

        print("Files selected for loading:")
        for fname in selected:
            print(" -", fname)

        # Here you can call your real loading function
        # load_heavy_data(fname)


def on_add_clicked(b):
    add_output.clear_output()
    with add_output:
        p = pathlib.Path(fc_path_input.selected_path).expanduser()

        if not p.exists():
            print("Path does not exist:", p)
            return

        global DF_FILES

        # Helper: check if filename already exists in df
        def already_in_df(path):
            return path.resolve() in DF_FILES["full_path"].values

        # Directory → add all files
        if p.is_dir():
            files = [f for f in p.glob("*") if f.is_file() and f.suffix in ['.h5', '.nxs']]
            new_rows = []

            for f in files:
                if already_in_df(f):
                    print(f"Skipping (already in table): {f.name}")
                else:
                    new_rows.append(file_metadata(f))

            if new_rows:
                DF_FILES = pandas.concat([DF_FILES, pandas.DataFrame(new_rows)], ignore_index=True)
                print(f"Added {len(new_rows)} new files from directory:", p)
            else:
                print("No new files to add from directory.")

        # Single file → add only if not already present
        elif p.is_file():
            if already_in_df(p):
                print("File already in table:", p.name)
            else:
                DF_FILES = pandas.concat([DF_FILES, pandas.DataFrame([file_metadata(p)])], ignore_index=True)
                print("Added file:", p.name)
        else:
            print("Path is neither file nor directory:", p)
            return

        refresh_selector()

In [ ]:
# Signals

fc_path_input.register_callback(on_add_clicked)
clear_button.on_click(on_clear_clicked)
# add_button.on_click(on_add_clicked)
load_button.on_click(on_load_clicked)

In [ ]:
# Display

ipywidgets.VBox([
    fc_path_input, 
    add_output,
    ipywidgets.HBox([file_selector, ipywidgets.VBox([load_button, clear_button])]),
    load_output
])

In [ ]:
widget_code_cell.make_code_cell(width="80%")